# Hybrid Quantum-Classical Siamese Network
Production-ready architecture for fingerprint verification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install qiskit qiskit-machine-learning qiskit-aer --upgrade -q
!pip install pylatexenc matplotlib opencv-python faiss-cpu -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os
import copy

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seeds set. Device:", torch.device("cpu"))


In [ ]:
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# 1. Precise Minutiae Labeling
def get_label(filename):
    # E.g., '10__M_Left_index_finger.BMP' -> '10_Left_index_finger'
    parts = filename.split("__")
    if len(parts) >= 2:
        return f"{parts[0]}_{parts[1][2:]}" # Skips the 'M_' or 'F_'
    return filename

def apply_clahe(img):
    img_np = np.array(img)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img_np)
    return Image.fromarray(img_clahe)

# 2. Self-Augmented Triplet Dataset
class TripletFingerprintDataset(Dataset):
    def __init__(self, root_dir, image_list, transform=None, num_triplets=1):
        self.root_dir = root_dir
        self.transform = transform
        self.triplets = self.build_triplets(image_list, num_triplets)
    
    def build_triplets(self, image_paths, num_triplets):
        label_to_images = {}
        for f in image_paths:
            label = get_label(f)
            label_to_images.setdefault(label, []).append(f)
        
        labels = list(label_to_images.keys())
        triplets = []
        for label in labels:
            imgs = label_to_images[label]
            if len(imgs) == 0: continue
            
            for _ in range(num_triplets):
                # Pick the exact same file for Anchor and Positive (Because SOCOFing Real only has 1 per finger)
                a = random.choice(imgs)
                p = a 
                
                # Pick a Negative from a definitively different Minutiae Class
                neg_label = random.choice([l for l in labels if l != label])
                n = random.choice(label_to_images[neg_label])
                triplets.append((a, p, n))
        return triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a_n, p_n, n_n = self.triplets[idx]
        
        # Load Raw Anchor Image
        img_a = Image.open(os.path.join(self.root_dir, a_n)).convert("L")
        img_n = Image.open(os.path.join(self.root_dir, n_n)).convert("L")
        
        # Apply heavy random transforms independently to generate visually distinct matrices from the identical physical file
        anchor = self.transform(img_a) if self.transform else img_a
        positive = self.transform(img_a) if self.transform else img_a
        negative = self.transform(img_n) if self.transform else img_n
        
        return anchor, positive, negative

# 3. Self-Augmented Siamese Dataset
def build_pairs(image_paths, num_pos_pairs=1):
    label_to_images = {}
    for f in image_paths:
        label = get_label(f)
        label_to_images.setdefault(label, []).append(f)
    
    labels = list(label_to_images.keys())
    pairs = []
    
    # Positive Logic: Pass the identical filename, flag 1
    for label in labels:
        imgs = label_to_images[label]
        for _ in range(num_pos_pairs):
            img = random.choice(imgs)
            pairs.append((img, img, 1))
            
    # Negative Logic: Distinct filenames, flag 0
    num_pos = len(pairs)
    for _ in range(num_pos):
        l1, l2 = random.sample(labels, 2)
        img1 = random.choice(label_to_images[l1])
        img2 = random.choice(label_to_images[l2])
        pairs.append((img1, img2, 0))
        
    random.shuffle(pairs)
    return pairs

class SiameseFingerprintDataset(Dataset):
    def __init__(self, root_dir, image_list, transform=None, num_pos_pairs=1):
        self.root_dir = root_dir
        self.transform = transform
        self.pairs = build_pairs(image_list, num_pos_pairs=num_pos_pairs)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_name, img2_name, label = self.pairs[idx]
        img1 = Image.open(os.path.join(self.root_dir, img1_name)).convert("L")
        img2 = Image.open(os.path.join(self.root_dir, img2_name)).convert("L")
        
        if self.transform:
            # Independent physical transform sweeps
            tensor1 = self.transform(img1)
            tensor2 = self.transform(img2)
            
        return tensor1, tensor2, torch.tensor(label, dtype=torch.float32)

# 95% Preprocessing Hardening
train_transform = transforms.Compose([
    transforms.Lambda(apply_clahe),
    transforms.RandomRotation(15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Lambda(apply_clahe),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

COLAB_PATH = '/content/drive/MyDrive/SOCOFing/Real'
LOCAL_PATH = os.path.join(os.getcwd(), 'dataset', 'Real')
dataset_path = COLAB_PATH if os.path.exists(COLAB_PATH) else LOCAL_PATH
if not os.path.exists(dataset_path): dataset_path = './'

all_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(".bmp")]
unique_labels = list(set([get_label(f) for f in all_files]))
random.shuffle(unique_labels)
n_train, n_val = int(0.7 * len(unique_labels)), int(0.15 * len(unique_labels))

train_labels = set(unique_labels[:n_train])
val_labels = set(unique_labels[n_train:n_train+n_val])
test_labels = set(unique_labels[n_train+n_val:])

# Hybrid Loading Architecture: Triplet for Matrix pushes, Pairs for Evaluation metrics
train_dataset = TripletFingerprintDataset(dataset_path, [f for f in all_files if get_label(f) in train_labels], transform=train_transform)
val_dataset = SiameseFingerprintDataset(dataset_path, [f for f in all_files if get_label(f) in val_labels], transform=val_transform)
test_dataset = SiameseFingerprintDataset(dataset_path, [f for f in all_files if get_label(f) in test_labels], transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, prefetch_factor=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, prefetch_factor=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, prefetch_factor=2, pin_memory=True)

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

device = torch.device("cpu")

def make_resnet(num_outputs=24):
    resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    for name, param in resnet.named_parameters():
        if 'layer4' in name or 'fc' in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    
    # Modify for Grayscale (1 channel instead of 3)
    resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    original_conv1 = models.resnet18(weights=ResNet18_Weights.DEFAULT).conv1.weight.data
    new_conv1 = original_conv1.mean(dim=1, keepdim=True)
    resnet.conv1.weight.data = new_conv1
    
    # Cap output feature space
    resnet.fc = nn.Linear(512, num_outputs)
    return resnet.to(device)

# Provide fallback legacy resnet reference if any code needs it, otherwise rely on function
resnet = make_resnet(12)
print("ResNet18 Factory Initialized.")

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

num_qubits = 8
qc = QuantumCircuit(num_qubits)
input_params = [Parameter(f"x{i}") for i in range(24)] 
weights_p    = [Parameter(f"w{i}") for i in range(8 * 3 * 1)]

def add_sel_layer(qc, p_params, input_data, layer_idx):
    for i in range(num_qubits):
        qc.rx(input_data[3*i], i)
        qc.ry(input_data[3*i+1], i)
        qc.rz(input_data[3*i+2], i)
    offset = layer_idx * 24
    for i in range(num_qubits):
        qc.rz(p_params[offset + 3*i], i)
        qc.ry(p_params[offset + 3*i + 1], i)
        qc.rz(p_params[offset + 3*i + 2], i)
    for i in range(num_qubits):
        for j in range(i + 1, num_qubits):
            qc.cx(i, j)

for L in range(1):
    add_sel_layer(qc, weights_p, input_params, L)

# Weighted Sum Observable (Voter System)
# Average of individual Z measurements: 1/8 * (Z0 + Z1 + ... + Z7)
obs_list = [("I" * i + "Z" + "I" * (num_qubits - 1 - i), 1/num_qubits) for i in range(num_qubits)]
observable = SparsePauliOp.from_list(obs_list)

qnn = EstimatorQNN(
    circuit=qc,
    estimator=AerEstimator(options={"backend_options": {"max_parallel_experiments": 0}}), 
    observables=observable,
    input_params=input_params, 
    weight_params=weights_p
)
quantum_layer = TorchConnector(qnn)
print("Hardened 8-Qubit SEL Circuit Ready (Weighted Sum Observable)")

In [ ]:
class HybridQNNModel(nn.Module):
    def __init__(self, base_model, q_layer):
        super().__init__()
        self.base = base_model
        self.qnn = q_layer
        self.scale = nn.Tanh()

    def _quantum_forward(self, f1, f2):
        # 1. Compute squared difference of raw features (Preserves relative scale + removes sharp non-differentiable V-shape)
        diff = (f1 - f2)**2
        # 2. Squash to [0, pi]
        diff_scaled = self.scale(diff) * torch.pi
        
        # 3. Predict via Quantum Simulator
        raw = self.qnn(diff_scaled)
        
        # 4. Normalize average Z expectation value from [-1, 1] to [0, 1]
        return ((raw + 1) / 2.0).squeeze()

    def forward(self, anchor, pos=None, neg=None):
        if pos is not None and neg is not None:
            # Triplet Training Mode
            f_a = self.base(anchor)
            f_p = self.base(pos)
            f_n = self.base(neg)
            
            sim_ap = self._quantum_forward(f_a, f_p)
            sim_an = self._quantum_forward(f_a, f_n)
            return sim_ap, sim_an
            
        elif pos is not None:
            # Pair Validation/Inference Mode
            f_a = self.base(anchor)
            f_p = self.base(pos)
            return self._quantum_forward(f_a, f_p)

In [ ]:
q_model = HybridQNNModel(make_resnet(num_outputs=24), quantum_layer).to(device)
optimizer = torch.optim.Adam(q_model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001, 
                                               steps_per_epoch=len(train_loader), epochs=10)
margin = 0.5
best_val_loss = float('inf')
train_losses_q = []
val_losses_q = []

print("Starting 95% Accuracy Quantum-Aware Training Loop...")
for epoch in range(3):
    # --- TRAINING PHASE ---
    q_model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/10 [Train]")
    for a, p, n in pbar:
        a, p, n = a.to(device), p.to(device), n.to(device)
        optimizer.zero_grad()
        
        # Get Quantum Similarities (Forces qiskit circuit to calculate gradients)
        sim_ap, sim_an = q_model(a, p, n)
        
        # Custom Triplet Loss using similarity metrics (Higher is better for positive)
        # We want: sim_an - sim_ap + margin <= 0
        loss = torch.mean(torch.clamp(sim_an - sim_ap + margin, min=0.0))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(q_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses_q.append(avg_train_loss)
    
    # --- VALIDATION PHASE ---
    q_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        # Using val_loader which emits (img1, img2, label) pairs
        for img1, img2, label in val_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            sim = q_model(img1, img2)
            
            # Reconstruct dummy validation loss (BCE or Contrastive) just for EER tracking
            # Using simple contrastive definition
            v_loss = torch.mean(label * (1 - sim)**2 + (1 - label) * torch.clamp(sim - 0.3, min=0)**2)
            total_val_loss += v_loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses_q.append(avg_val_loss)
    
    print(f"Epoch {epoch+1} Completed: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Checkpoint safe saving
        torch.save(q_model.state_dict(), '/content/drive/MyDrive/SOCOFing/hybrid_95plus_best.pth')
        print("  >> Saved new Best Model to Drive")

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(train_losses_q, label='Quantum Train Loss')
plt.plot(val_losses_q, label='Quantum Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Quantum Model Training')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d

# Fatal Bug Fixed: Target new High-Accuracy trained weights
q_model.load_state_dict(torch.load('/content/drive/MyDrive/SOCOFing/hybrid_95plus_best.pth', map_location=device))
q_model.eval()
q_scores = []
q_labels = []

with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        sim = q_model(img1, img2)
        q_scores.extend(sim.cpu().numpy().flatten())
        q_labels.extend(label.cpu().numpy().flatten())

fpr_q, tpr_q, thresholds_q = roc_curve(q_labels, q_scores)
roc_auc_q = auc(fpr_q, tpr_q)
eer_q = brentq(lambda x: 1. - x - interp1d(fpr_q, tpr_q)(x), 0., 1.)
thresh_q = interp1d(fpr_q, thresholds_q)(eer_q)
preds_q = (np.array(q_scores) >= thresh_q).astype(int)

print(f"\n=== Quantum Evaluation ===")
print(f"AUC: {roc_auc_q:.4f} | EER: {eer_q:.4f} (Thresh: {thresh_q:.3f})")
print(f"Accuracy:  {accuracy_score(q_labels, preds_q)*100:.2f}%")
print(f"Precision: {precision_score(q_labels, preds_q, zero_division=0)*100:.2f}%")
print(f"Recall:    {recall_score(q_labels, preds_q, zero_division=0)*100:.2f}%")
print(f"F1-Score:  {f1_score(q_labels, preds_q, zero_division=0)*100:.2f}%")

In [ ]:
class ClassicalSiamese(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.classical_head = nn.Sequential(
            nn.Linear(8, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x1, x2):
        f1 = self.base(x1)
        f2 = self.base(x2)
        diff = torch.abs(f1 - f2)
        multi = f1 * f2
        combined = torch.cat((diff, multi), dim=1)
        return self.classical_head(combined).squeeze()

c_model = ClassicalSiamese(copy.deepcopy(resnet)).to(device)

# Unfreeze deeper layers explicitly on the classical copy (no effect to quantum)
for param in c_model.base.layer2.parameters():
    param.requires_grad = True
for param in c_model.base.layer3.parameters():
    param.requires_grad = True

criterion_c = nn.BCEWithLogitsLoss()

optimizer_c = torch.optim.Adam([
    {'params': c_model.classical_head.parameters(), 'lr': 0.001},
    {'params': filter(lambda p: p.requires_grad, c_model.base.parameters()), 'lr': 0.0001}
], weight_decay=1e-5)

epochs = 5
patience = 2
best_val_loss_c = float('inf')
wait = 0
train_losses_c = []
val_losses_c = []

for epoch in range(epochs):
    c_model.train()
    total_train_loss = 0
    pbar = tqdm(classic_train_loader, desc=f"Epoch {epoch+1}/{epochs} (Classic Train)")
    for img1, img2, label in pbar:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer_c.zero_grad()
        sim = c_model(img1, img2)
        loss = criterion_c(sim, label)
        loss.backward()
        clip_grad_norm_(c_model.parameters(), max_norm=1.0)
        optimizer_c.step()
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_train_loss / len(classic_train_loader)
    train_losses_c.append(avg_train_loss)

    c_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for img1, img2, label in val_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            sim = c_model(img1, img2)
            loss = criterion_c(sim, label)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses_c.append(avg_val_loss)
    print(f"Classic Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss_c:
        best_val_loss_c = avg_val_loss
        wait = 0
        torch.save(c_model.state_dict(), '/content/drive/MyDrive/SOCOFing/classic_siamese_best.pth')
        print("   >> Saved best model")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

plt.figure(figsize=(8,4))
plt.plot(train_losses_c, label='Classic Train Loss')
plt.plot(val_losses_c, label='Classic Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Classical Model Training')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# 1. Precise Minutiae Labeling
def get_label(filename):
    # E.g., '10__M_Left_index_finger.BMP' -> '10_Left_index_finger'
    parts = filename.split("__")
    if len(parts) >= 2:
        return f"{parts[0]}_{parts[1][2:]}" # Skips the 'M_' or 'F_'
    return filename

def apply_clahe(img):
    img_np = np.array(img)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img_np)
    return Image.fromarray(img_clahe)

# 2. Self-Augmented Triplet Dataset
class TripletFingerprintDataset(Dataset):
    def __init__(self, root_dir, image_list, transform=None, num_triplets=1):
        self.root_dir = root_dir
        self.transform = transform
        self.triplets = self.build_triplets(image_list, num_triplets)
    
    def build_triplets(self, image_paths, num_triplets):
        label_to_images = {}
        for f in image_paths:
            label = get_label(f)
            label_to_images.setdefault(label, []).append(f)
        
        labels = list(label_to_images.keys())
        triplets = []
        for label in labels:
            imgs = label_to_images[label]
            if len(imgs) == 0: continue
            
            for _ in range(num_triplets):
                # Pick the exact same file for Anchor and Positive (Because SOCOFing Real only has 1 per finger)
                a = random.choice(imgs)
                p = a 
                
                # Pick a Negative from a definitively different Minutiae Class
                neg_label = random.choice([l for l in labels if l != label])
                n = random.choice(label_to_images[neg_label])
                triplets.append((a, p, n))
        return triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a_n, p_n, n_n = self.triplets[idx]
        
        # Load Raw Anchor Image
        img_a = Image.open(os.path.join(self.root_dir, a_n)).convert("L")
        img_n = Image.open(os.path.join(self.root_dir, n_n)).convert("L")
        
        # Apply heavy random transforms independently to generate visually distinct matrices from the identical physical file
        anchor = self.transform(img_a) if self.transform else img_a
        positive = self.transform(img_a) if self.transform else img_a
        negative = self.transform(img_n) if self.transform else img_n
        
        return anchor, positive, negative

# 3. Self-Augmented Siamese Dataset
def build_pairs(image_paths, num_pos_pairs=1):
    label_to_images = {}
    for f in image_paths:
        label = get_label(f)
        label_to_images.setdefault(label, []).append(f)
    
    labels = list(label_to_images.keys())
    pairs = []
    
    # Positive Logic: Pass the identical filename, flag 1
    for label in labels:
        imgs = label_to_images[label]
        for _ in range(num_pos_pairs):
            img = random.choice(imgs)
            pairs.append((img, img, 1))
            
    # Negative Logic: Distinct filenames, flag 0
    num_pos = len(pairs)
    for _ in range(num_pos):
        l1, l2 = random.sample(labels, 2)
        img1 = random.choice(label_to_images[l1])
        img2 = random.choice(label_to_images[l2])
        pairs.append((img1, img2, 0))
        
    random.shuffle(pairs)
    return pairs

class SiameseFingerprintDataset(Dataset):
    def __init__(self, root_dir, image_list, transform=None, num_pos_pairs=1):
        self.root_dir = root_dir
        self.transform = transform
        self.pairs = build_pairs(image_list, num_pos_pairs=num_pos_pairs)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_name, img2_name, label = self.pairs[idx]
        img1 = Image.open(os.path.join(self.root_dir, img1_name)).convert("L")
        img2 = Image.open(os.path.join(self.root_dir, img2_name)).convert("L")
        
        if self.transform:
            # Independent physical transform sweeps
            tensor1 = self.transform(img1)
            tensor2 = self.transform(img2)
            
        return tensor1, tensor2, torch.tensor(label, dtype=torch.float32)

# 95% Preprocessing Hardening
train_transform = transforms.Compose([
    transforms.Lambda(apply_clahe),
    transforms.RandomRotation(15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Lambda(apply_clahe),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

COLAB_PATH = '/content/drive/MyDrive/SOCOFing/Real'
LOCAL_PATH = os.path.join(os.getcwd(), 'dataset', 'Real')
dataset_path = COLAB_PATH if os.path.exists(COLAB_PATH) else LOCAL_PATH
if not os.path.exists(dataset_path): dataset_path = './'

all_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(".bmp")]
unique_labels = list(set([get_label(f) for f in all_files]))
random.shuffle(unique_labels)
n_train, n_val = int(0.7 * len(unique_labels)), int(0.15 * len(unique_labels))

train_labels = set(unique_labels[:n_train])
val_labels = set(unique_labels[n_train:n_train+n_val])
test_labels = set(unique_labels[n_train+n_val:])

# Hybrid Loading Architecture: Triplet for Matrix pushes, Pairs for Evaluation metrics
train_dataset = TripletFingerprintDataset(dataset_path, [f for f in all_files if get_label(f) in train_labels], transform=train_transform)
val_dataset = SiameseFingerprintDataset(dataset_path, [f for f in all_files if get_label(f) in val_labels], transform=val_transform)
test_dataset = SiameseFingerprintDataset(dataset_path, [f for f in all_files if get_label(f) in test_labels], transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, prefetch_factor=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, prefetch_factor=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, prefetch_factor=2, pin_memory=True)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

num_qubits = 8
qc = QuantumCircuit(num_qubits)
input_params = [Parameter(f"x{i}") for i in range(24)] 
weights_p    = [Parameter(f"w{i}") for i in range(8 * 3 * 1)]

def add_sel_layer(qc, p_params, input_data, layer_idx):
    for i in range(num_qubits):
        qc.rx(input_data[3*i], i)
        qc.ry(input_data[3*i+1], i)
        qc.rz(input_data[3*i+2], i)
    offset = layer_idx * 24
    for i in range(num_qubits):
        qc.rz(p_params[offset + 3*i], i)
        qc.ry(p_params[offset + 3*i + 1], i)
        qc.rz(p_params[offset + 3*i + 2], i)
    for i in range(num_qubits):
        for j in range(i + 1, num_qubits):
            qc.cx(i, j)

for L in range(1):
    add_sel_layer(qc, weights_p, input_params, L)

# Weighted Sum Observable (Voter System)
# Average of individual Z measurements: 1/8 * (Z0 + Z1 + ... + Z7)
obs_list = [("I" * i + "Z" + "I" * (num_qubits - 1 - i), 1/num_qubits) for i in range(num_qubits)]
observable = SparsePauliOp.from_list(obs_list)

qnn = EstimatorQNN(
    circuit=qc,
    estimator=AerEstimator(options={"backend_options": {"max_parallel_experiments": 0}}), 
    observables=observable,
    input_params=input_params, 
    weight_params=weights_p
)
quantum_layer = TorchConnector(qnn)
print("Hardened 8-Qubit SEL Circuit Ready (Weighted Sum Observable)")

In [ ]:
# Visualizing the 4-Qubit Quantum Circuit used in the Hybrid Model
import matplotlib.pyplot as plt
qc.draw(output='mpl')
plt.show()
